# Felix's OLS model — code handoff

Three signals: one-month momentum, front-minus-second-month carry, and lagged dollar return. Reads Mike's current shared cleaned dataset. **Run All fits the training sample only; it does not evaluate the validation or test windows.** See `models/Benchmark/handoff/README.md` for original code and the two earlier experimental protocols.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/clean/oil_monthly_clean.csv').exists())
SOURCE = ROOT / 'data/clean/oil_monthly_clean.csv'
data = pd.read_csv(SOURCE, parse_dates=['date']).set_index('date').sort_index()
assert data.index.is_unique and data.index.is_month_end.all()
assert data.index.equals(pd.date_range(data.index.min(), data.index.max(), freq='ME'))
print('Shared input: data/clean/oil_monthly_clean.csv')

Shared input: data/clean/oil_monthly_clean.csv


## Definitions at forecast origin t

- Target: next month's CL1 percentage change, `100*(CL1(t+1)/CL1(t)-1)`.
- Momentum: this month's CL1 percentage change; **one-month window**.
- Carry: `100*(CL1(t)-CL2(t))/CL1(t)`; positive = backwardation. This is **minus Mike's carry column**.
- Dollar: `100*dxy_ret(t-1)`; one extra month of delay, expressed in percentage points.

All coefficients are estimated by OLS with an intercept. No variable selection or scaling.

In [2]:
FEATURES = ['momentum', 'carry', 'dollar']
frame = pd.DataFrame(index=data.index)
frame['target_month'] = data.target_month
frame['sample'] = data['sample']
frame['actual'] = data.y
frame['momentum'] = data.cl1_pct
frame['carry'] = 100*(data.cl1-data.cl2)/data.cl1
frame['dollar'] = 100*data.dxy_ret.shift(1)
# The shared CSV is rounded to six decimal places.
np.testing.assert_allclose(frame['carry'], -data.carry, atol=2e-6, rtol=0)
np.testing.assert_allclose(frame.momentum.iloc[1:], (100*data.cl1.pct_change()).iloc[1:], atol=2e-6, rtol=0)
assert (pd.PeriodIndex(frame.target_month, freq='M') == data.index.to_period('M')+1).all()
print('Feature definitions and target-month alignment: PASS')

Feature definitions and target-month alignment: PASS


In [3]:
train = frame.loc[frame['sample'].eq('train')].dropna(subset=['actual']+FEATURES)
assert (pd.PeriodIndex(train.target_month, freq='M') <= pd.Period('2004-12')).all()
fit = sm.OLS(train.actual, sm.add_constant(train[FEATURES], has_constant='add')).fit()
print(f'Training targets: {train.target_month.iloc[0]} to {train.target_month.iloc[-1]} ({len(train)} rows)')
display(fit.params.rename('coefficient').to_frame())

Training targets: 1995-02 to 2004-12 (119 rows)


,coefficient
const,1.756784
momentum,-0.051880
carry,-0.603438
dollar,0.278033


## Hand over to the group evaluator

`frame` contains origin-indexed features, target months, sample labels and actual outcomes. The group's evaluator can use these definitions for its agreed validation and final testing. Do not apply another lag to momentum or carry: the table is already indexed by forecast origin.

Earlier Felix experiments have already examined overlapping test years; this notebook makes no untouched-test claim. The model forecasts continuous CL1 changes, not verified executable futures profits.